# Hindi SLM — Inference Notebook

**Pipeline:**  
English prompt → Ollama (Gemma) translation → Hindi SLM (`vaibhavmaurya/hindi-slm-v001`) → Generated Hindi text

Each cell is independent and clearly labelled. Token counts (input + output) are shown for every inference call.

## Cell 1 — Install dependencies

Run once. Skip if already installed.

In [1]:
%pip install -q transformers accelerate torch ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Load tokenizer from HuggingFace Hub

In [2]:
from transformers import AutoTokenizer

REPO_ID = "vaibhavmaurya/hindi-slm-v001"

tokenizer = AutoTokenizer.from_pretrained(REPO_ID)

print(f"Tokenizer loaded")
print(f"  Vocab size : {tokenizer.vocab_size:,}")
print(f"  BOS token  : {tokenizer.bos_token!r}  (id={tokenizer.bos_token_id})")
print(f"  EOS token  : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")

Tokenizer loaded
  Vocab size : 32,000
  BOS token  : '<s>'  (id=2)
  EOS token  : '</s>'  (id=3)


## Cell 3 — Load Hindi SLM model from HuggingFace Hub

In [3]:
import torch
from transformers import AutoModelForCausalLM

# Use bfloat16 on CUDA, float32 on CPU
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype=dtype,
    device_map="auto",
)
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"  Device     : {device.upper()}")
print(f"  Dtype      : {dtype}")
print(f"  Parameters : {param_count / 1e6:.2f}M")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  GPU        : {torch.cuda.get_device_name(0)}  ({vram:.1f} GB)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/93 [00:00<?, ?it/s]

Model loaded
  Device     : CUDA
  Dtype      : torch.bfloat16
  Parameters : 62.93M
  GPU        : NVIDIA RTX 3000 Ada Generation Laptop GPU  (8.0 GB)


## Cell 4 — Ollama translation helper (English → Hindi)

Uses `gemma3:1b` by default — the lightest Gemma model Ollama supports.  
Change `OLLAMA_MODEL` to `gemma3:4b`, `gemma2:2b`, or any model you have pulled.

**Pre-requisite:** Ollama must be running (`ollama serve`) and the model must be pulled:
```
ollama pull gemma3:1b
```

In [5]:
import ollama

OLLAMA_MODEL = "llama3.2:latest"   # change to gemma3:1b, gemma3:4b, gemma2:2b, etc.

def translate_to_hindi(english_text: str) -> dict:
    """
    Translate English text to Hindi using Ollama.
    Returns dict with: hindi_text, input_tokens, output_tokens.
    """
    system_prompt = (
        "You are a precise English-to-Hindi translator. "
        "Translate the user's sentence into natural, fluent Hindi using Devanagari script. "
        "Output ONLY the Hindi translation — no explanations, no Roman script, no punctuation changes."
    )

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": english_text},
        ],
    )

    hindi = response["message"]["content"].strip()
    usage = response.get("usage", {})

    # Ollama's Python client exposes token counts in response.usage (newer versions)
    # Fall back to word-count estimate if not available
    in_tokens  = usage.get("prompt_tokens",     len(english_text.split()))
    out_tokens = usage.get("completion_tokens", len(hindi.split()))

    return {
        "hindi_text":   hindi,
        "input_tokens":  in_tokens,
        "output_tokens": out_tokens,
    }

# Sanity check
test = translate_to_hindi("Hello, how are you? How is going in life. What do you know about India")
print(f"Translation  : {test['hindi_text']}")
print(f"Input tokens : {test['input_tokens']}")
print(f"Output tokens: {test['output_tokens']}")

Translation  : नमस्ते, आप कैसे हैं? जीवन में आपका क्या हाल है। भारत के बारे में आपको क्या पता है?
Input tokens : 15
Output tokens: 18


## Cell 5 — Hindi SLM inference helper

Runs the translated Hindi through `vaibhavmaurya/hindi-slm-v001` and reports exact token counts using the model's own tokenizer.

In [6]:
import time

def infer_hindi_slm(
    hindi_prompt: str,
    max_new_tokens: int = 500,
    temperature: float = 0.8,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> dict:
    """
    Run hindi_prompt through the Hindi SLM.
    Returns dict with: full_text, generated_text, input_tokens, output_tokens, elapsed_sec.
    """
    inputs = tokenizer(hindi_prompt, return_tensors="pt").to(model.device)
    input_token_count = inputs["input_ids"].shape[1]

    # print(f'''
    # inputs:
    # {inputs}
    # ''')
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0

    # output_ids includes the prompt; slice to get only new tokens
    # print(f'''
    # output_ids:
    # {output_ids}
    # ''')
    generated_ids    = output_ids[0, input_token_count:]
    output_token_count = generated_ids.shape[0]

    full_text      = tokenizer.decode(output_ids[0],    skip_special_tokens=True)
    generated_text = tokenizer.decode(generated_ids,    skip_special_tokens=True)

    return {
        "full_text":      full_text,
        "generated_text": generated_text,
        "input_tokens":   input_token_count,
        "output_tokens":  output_token_count,
        "elapsed_sec":    elapsed,
    }

print("infer_hindi_slm() ready.")

infer_hindi_slm() ready.


## Cell 6 — Full pipeline: English → Gemma (translate) → Hindi SLM (generate)

Edit `ENGLISH_PROMPT` and run. Everything else is automatic.

In [7]:
# ── Change this prompt ────────────────────────────────────────────────────────
ENGLISH_PROMPT = "The capital of India is New Delhi, and it is a very ancient city. What do you know about Uttar Pradesh"
# ─────────────────────────────────────────────────────────────────────────────

SEP = "─" * 68

# Step 1: Translate with Ollama / Gemma
print(SEP)
print("  Step 1 — Translation  (Ollama / Gemma)")
print(SEP)
translation = translate_to_hindi(ENGLISH_PROMPT)

print(f"  English  : {ENGLISH_PROMPT}")
print(f"  Hindi    : {translation['hindi_text']}")
print(f"  Tokens   : {translation['input_tokens']} in  →  {translation['output_tokens']} out")

# Step 2: Infer with Hindi SLM
print()
print(SEP)
print("  Step 2 — Hindi SLM inference")
print(SEP)
result = infer_hindi_slm(translation["hindi_text"])

print(f"  Prompt (translated) : {translation['hindi_text']}")
print()
print(f"  Generated text:")
print(f"  {result['generated_text']}")
print()
print(f"  Input tokens  : {result['input_tokens']}")
print(f"  Output tokens : {result['output_tokens']}")
print(f"  Total tokens  : {result['input_tokens'] + result['output_tokens']}")
print(f"  Time          : {result['elapsed_sec']:.2f}s  "
      f"({result['output_tokens'] / result['elapsed_sec']:.1f} tok/s)")
print()
print(SEP)
print("  Full output (prompt + continuation):")
print(SEP)
print(f"  {result['full_text']}")

────────────────────────────────────────────────────────────────────
  Step 1 — Translation  (Ollama / Gemma)
────────────────────────────────────────────────────────────────────
  English  : The capital of India is New Delhi, and it is a very ancient city. What do you know about Uttar Pradesh
  Hindi    : भारत की राजधानी नई दिल्ली है और यह बहुत प्राचीन शहर है। उत्तर प्रदेश के बारे में तो मुझे पता है कि यह भारत का सबसे बड़ा राज्य है जिसमें विभिन्न संस्कृतियों और भाषाओं के लोग रहते हैं।
  Tokens   : 21 in  →  38 out

────────────────────────────────────────────────────────────────────
  Step 2 — Hindi SLM inference
────────────────────────────────────────────────────────────────────

    inputs:
    {'input_ids': tensor([[  101,    11,   930,   264,   195,    20,    14,    35,   148,  1884,
           404,    17,   338,   176,     8,   186,     9,    38,   374,   335,
            20,    21,    35,   101,    16,   160,   390,   174,    20,   203,
           405, 21871,    14,  4585,     

[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (512). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.



    output_ids:
    tensor([[  101,    11,   930,   264,   195,    20,    14,    35,   148,  1884,
           404,    17,   338,   176,     8,   186,     9,    38,   374,   335,
            20,    21,    35,   101,    16,   160,   390,   174,    20,   203,
           405, 21871,    14,  4585,     8,   141,   604,    28,    26,   103,
            19,  3796,   129,    20,    21,    95,   872,    24,    13,  2167,
            24,   807,  2512,  5858,    28,   147,    95,   195,     8,   986,
            14,  1854,     8,   186,     9,   922,    50,    14,   289,   154,
            23,   529,    17,   374,    10,  6848,    93,    21,   147,   195,
            12,    10,  3220,    13,   878,    33,    63,    20,   263,   195,
            16,   123,   722,    13,    31,   911,   522,    99,    72,    75,
            35,    23,   148,   162,    31,    80,    18,   474,   477,    21,
           195,    16,   404,   195,    13,   148,   447,    17,   195,    16,
          2220,  1762,   872,  

## Cell 7 — Batch: run multiple English prompts at once

In [7]:
PROMPTS = [
    "Today's news is about the Indian economy growing rapidly.",
    "Once upon a time, in a dense forest, there lived a wise old owl.",
    "Hello! How are you? I hope you are doing well.",
    "The moonlight reflected off the river, creating a silver spectacle.",
    "India gained independence on 15th August 1947 under Mahatma Gandhi's leadership.",
]

SEP = "─" * 68
total_in = total_out = 0

for i, eng in enumerate(PROMPTS, 1):
    print(f"\n{'═' * 68}")
    print(f"  Prompt {i} / {len(PROMPTS)}")
    print(f"{'═' * 68}")

    # Translate
    tr = translate_to_hindi(eng)
    print(f"  EN : {eng}")
    print(f"  HI : {tr['hindi_text']}")
    print(f"  Translation tokens — in: {tr['input_tokens']}  out: {tr['output_tokens']}")

    # Infer
    res = infer_hindi_slm(tr["hindi_text"], max_new_tokens=100)
    print(f"\n  SLM continuation:")
    print(f"  {res['generated_text']}")
    print(f"\n  SLM tokens — in: {res['input_tokens']}  out: {res['output_tokens']}"
          f"  total: {res['input_tokens'] + res['output_tokens']}"
          f"  ({res['output_tokens'] / res['elapsed_sec']:.1f} tok/s)")

    total_in  += res["input_tokens"]
    total_out += res["output_tokens"]

print(f"\n{'═' * 68}")
print(f"  Batch summary")
print(f"{'═' * 68}")
print(f"  Prompts processed : {len(PROMPTS)}")
print(f"  Total input tokens (SLM)  : {total_in}")
print(f"  Total output tokens (SLM) : {total_out}")
print(f"  Grand total tokens        : {total_in + total_out}")


════════════════════════════════════════════════════════════════════
  Prompt 1 / 5
════════════════════════════════════════════════════════════════════


  EN : Today's news is about the Indian economy growing rapidly.
  HI : आजकल की खबरें भारतीय अर्थव्यवस्था तेजी से बढ़ रही है।
  Translation tokens — in: 9  out: 10



  SLM continuation:
  जैसे-जैसे अर्थव्यवस्था में मंदी आ रही है वैसे-वैसे देश में महंगाई बढ़ती जा रही है और लोग अपने देश के लिए कुछ भी कर रहे हैं। यही कारण है कि महंगाई दर का आंकड़ा लगातार बढ़ता जा रहा है। हाल ही में केंद्र सरकार ने पेट्रोल और डीजल पर अतिरिक्त ड्यूटी में कटौती करने का ऐलान किया है। सरकार ने पेट्रोल और डीजल पर प्रति लीटर 7 रुपये की बढ़ोतरी कर दी है। इसके अलावा अन्य चीजों पर भी लागू की गई हैं। इससे देश के लिए ईंधन की कीमतों में काफी उछाल आया है।

  SLM tokens — in: 10  out: 98  total: 108  (32.1 tok/s)

════════════════════════════════════════════════════════════════════
  Prompt 2 / 5
════════════════════════════════════════════════════════════════════


  EN : Once upon a time, in a dense forest, there lived a wise old owl.
  HI : कभी कुछ समय पहले, एक घने जंगल में, एक बुद्धि वाला पुराना हंस रहता था।
  Translation tokens — in: 14  out: 15



  SLM continuation:
  उसने एक बार कहा कि वह हमेशा अच्छी तरह से वाकिफ नहीं था और कुछ भी अच्छा नहीं था। उसने अपने दिल की बात कही कि उसने उसे कभी एक छोटे से गांव में नहीं देखा था। उसने सोचा कि यह बहुत मुश्किल है और मुझे कभी भी उससे प्यार नहीं हुआ था। लेकिन ऐसा करने के बाद, वह बहुत खुश था, और एक बार उसने मुझे उससे प्यार किया! एबोक ने कहा, "सर, अब मैं तुम्हें बता देता हूं कि तुम मुझे जानती हो? यह अच्छा है। " उसने

  SLM tokens — in: 16  out: 100  total: 116  (35.9 tok/s)

════════════════════════════════════════════════════════════════════
  Prompt 3 / 5
════════════════════════════════════════════════════════════════════


...


  SLM continuation:
  तुम मुझे बता दोगे ।

  SLM tokens — in: 14  out: 9  total: 23  (31.6 tok/s)

════════════════════════════════════════════════════════════════════
  Prompt 4 / 5
════════════════════════════════════════════════════════════════════


  EN : The moonlight reflected off the river, creating a silver spectacle.
  HI : चंद्र की रोशनी नदी पर पड़ते हुए एक चांदी का स्थल बनाती है।
  Translation tokens — in: 10  out: 13



  SLM continuation:
  इस पर भगवान शिव का वास होता है। यह स्थान भगवान विष्णु के चरण माने जाते हैं। यह स्थान मूलतः आंध्र प्रदेश में स्थित है। इस स्थान पर शिव के साथ लक्ष्मी, गणेश, शिव, गणेश, लक्ष्मी और लक्ष्मी भी निवास करते हैं। भगवान शिव को प्रसन्न करने के लिए इन जगहों पर लाल रंग के फूल अर्पित किए जाते हैं। . एकेश्वर महादेव मंदिर भारत के उत्तराखण्ड राज्य में स्थित एक नगर है। इसका नाम श्रीनाथ मंदिर था। . गंगा नदी या कोई अन्य जल स्रोत (कोयला)

  SLM tokens — in: 13  out: 100  total: 113  (30.7 tok/s)

════════════════════════════════════════════════════════════════════
  Prompt 5 / 5
════════════════════════════════════════════════════════════════════


  EN : India gained independence on 15th August 1947 under Mahatma Gandhi's leadership.
  HI : भारत को १५ अगस्त १९४७ को महात्मा गांधी के नेतृत्व में स्वतंत्रता मिली।
  Translation tokens — in: 11  out: 13



  SLM continuation:
  यह भारत का सर्वाधिक लोकप्रिय राज्य है। . थलसेना (१८८६ - १८८०) की स्थापना १८८७ में हुई थी और यह स्वयं भारतीय सेना की एक महत्वपूर्ण शाखा थी। थलसेना के दौरान इसे २९,००० मील की दूरी पर अवस्थित किया गया। इसके प्रमुख अंग - बल, पराक्रम, युद्ध, शक्ति, युद्ध, युद्ध तथा नौसेना के विभिन्न अंगों में विस्तृत एवं महत्वपूर्ण स्थान था। भारतीय सेना का लक्ष्य था कि वह भारत को युद्ध के मैदान से अलग कर दे तथा उस क्षेत्र

  SLM tokens — in: 14  out: 100  total: 114  (31.4 tok/s)

════════════════════════════════════════════════════════════════════
  Batch summary
════════════════════════════════════════════════════════════════════
  Prompts processed : 5
  Total input tokens (SLM)  : 67
  Total output tokens (SLM) : 407
  Grand total tokens        : 474
